# Cleaning Airbnb Listings Dataset

[Inside Airbnb](https://insideairbnb.com/get-the-data/) provides datasets containing Airbnb listings for various cities around the world. These datasets often require cleaning before they can be effectively used for analysis. This notebook cleans the detailed listings file for **Barcelona, Spain, from the Inside Airbnb snapshot dated 17 September 2026**.

## Data Cleaning Steps

1. **Import Libraries**: We will use `pandas` for data manipulation and `numpy` for numerical operations.
2. **Load Dataset**: Read the compressed (gzip) CSV file containing the Airbnb listings data.
3. **Inspect Data**: Check the structure of the dataset and count the missing values in each column.
4. **Select Relevant Columns**: Identify and retain only the columns that are necessary for analysis.
5. **Data Type Conversion**: Ensure that each column has the appropriate data type.
   - The number of bathrooms is stored as text (for example, `"1.5 shared baths"`), so we extract the number.
   - The price column contains a currency symbol and commas, which need to be removed before conversion.
   - The superhost flag is stored as `"t"`/`"f"`, which we convert to 1/0 without losing missing values.
6. **Handle Missing Values**: Decide, column by column, whether to drop rows, fill values, or keep values missing.
7. **Validate**: Check that the cleaned values are in plausible ranges, and look at the extremes.
8. **Save**: Write the cleaned data to a CSV file.

▶️ Import libraries.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from urllib.request import urlretrieve

▶️ Display up to 100 columns in this Jupyter notebook.


In [2]:
pd.set_option("display.max_columns", 100)

:::{tip} Maximum Number of Rows
You can also set the maximum number of rows to display using the following code:

```python
pd.set_option("display.max_rows", 100)
```

If you want to display all columns without any limit, you can set it to `None`:

```python
pd.set_option("display.max_columns", None)
```

This is not recommended for very wide DataFrames, as it may clutter your output.

:::


:::{note} Other commonly used `pd.set_option()` settings

- `pd.set_option("display.width", 120)` sets the total console width.
- `pd.set_option("display.max_colwidth", 50)` sets the maximum column width for each column. This can be used to truncate long strings.
- `pd.set_option("display.precision", 2)` sets the number of decimal places for floating-point numbers.
- `pd.set_option("display.float_format", "{:.2f}".format)` formats floating-point numbers to two decimal places. This can be useful if you want to control the exact display format of floats.

:::


## Load Dataset


We use Inside Airbnb's detailed listings file for Barcelona from the snapshot dated 17 September 2026, saved in this case study's `data/` folder. Inside Airbnb keeps roughly the last 12 months of snapshots online, so a link to one specific snapshot eventually stops working. Reading a saved copy keeps this notebook reproducible.

If the file is missing, the code below downloads it once and saves it to `data/`. Inside Airbnb asks users to download its data once rather than every time an analysis runs.

The `compression="gzip"` parameter is used because the dataset file is compressed in gzip format. If your dataset is not compressed, you should omit this parameter. This parameter is optional if the file name ends with `.gz`, as pandas can automatically detect the compression format based on the file extension.

:::{note} Source and licence

Data: [Inside Airbnb](https://insideairbnb.com/get-the-data/), Barcelona, snapshot of 17 September 2026. Inside Airbnb states that its datasets "are available to download and use under the Creative Commons Attribution 4.0 International (CC BY 4.0) licence". See its [Data Policies](https://insideairbnb.com/data-policies/) page for how to credit the data.

:::

In [3]:
DATA_PATH = Path("data/listings.csv.gz")
SOURCE_URL = (
    "https://data.insideairbnb.com/spain/catalonia/barcelona/"
    "2026-09-17/data/listings.csv.gz"
)

if not DATA_PATH.exists():
    # Download the snapshot once and keep a local copy.
    DATA_PATH.parent.mkdir(exist_ok=True)
    urlretrieve(SOURCE_URL, DATA_PATH)

df = pd.read_csv(DATA_PATH, compression="gzip")
df.head(2)

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_profile_id,host_profile_url,host_name,host_since,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,price_quote_checkin_date,price_quote_checkout_date,price_quote_total_price,price_quote_price_per_night,price_quote_raw,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,18674,https://www.airbnb.com/rooms/18674,20260917041147,2026-09-18,NaN,Huge flat for 8 people close to Sagrada Familia,"* Currently, facade works are being carried ou...",NaN,https://a0.muscache.com/pictures/hosting/Hosti...,71615,https://www.airbnb.com/users/show/71615,1462508358342835057,https://www.airbnb.com/users/profile/146250835...,Maria Y Mireia Y Ana Barcelona4Seasons,NaN,16.0,7.0,15.0,8.0,"Barcelona, Spain","We are Mireia and Maria, two multilingual entr...",NaN,NaN,NaN,f,NaN,https://a0.muscache.com/im/pictures/user/User/...,NaN,32,NaN,NaN,t,t,NaN,la Sagrada Família,Eixample,41.40556,2.17262,Entire rental unit,Entire home/apt,8,NaN,2 baths,3.0,5.0,"[""Toaster"", ""AC - split type ductless system"",...",$274.00,2026-09-23,2026-09-25,548.0,274.00,"{""quote"": {""taxes"": null, ""currency"": ""EUR"", ""...",1.0,999.0,1.0,4.0,999.0,999.0,3.4,999.0,NaN,t,12,36,62,162,2026-09-18,60,8,0,77,9,48,13152.0,2013-05-27,2026-07-20,4.42,4.47,4.59,4.66,4.63,4.81,4.37,Spain - National registration number<br />ESFC...,NaN,21,21,0,0,0.37
1,23197,https://www.airbnb.com/rooms/23197,20260917041147,2026-09-18,NaN,CCIB Forum • Spacious 3BR • Large Balcony,"Beautiful, spacious apartment with large balco...",NaN,https://a0.muscache.com/pictures/hosting/Hosti...,90417,https://www.airbnb.com/users/show/90417,1462509351463595774,https://www.airbnb.com/users/profile/146250935...,Marnie,NaN,16.0,6.0,15.0,8.0,"Catalonia, Spain","Hi there,\n\nI’m marnie, originally from Austr...",NaN,NaN,NaN,t,NaN,https://a0.muscache.com/im/pictures/user/44b56...,NaN,5,NaN,NaN,t,t,NaN,el Besòs i el Maresme,Sant Martí,41.41243,2.21975,Entire rental unit,Entire home/apt,5,NaN,2 baths,3.0,4.0,"[""Toaster"", ""Portable heater"", ""AC - split typ...",$407.67,2026-09-20,2026-09-23,1223.0,407.67,"{""quote"": {""taxes"": null, ""currency"": ""EUR"", ""...",3.0,1125.0,3.0,4.0,1125.0,1125.0,3.0,1125.0,NaN,t,23,44,74,128,2026-09-18,102,11,2,89,11,66,26906.0,2011-03-15,2026-09-07,4.84,4.95,4.91,4.94,4.99,4.70,4.72,Spain - National registration number<br />ESFC...,NaN,1,1,0,0,0.54


▶️ Confirm which snapshot the file comes from. The `last_scraped` column records the date each listing was scraped.

In [4]:
df["last_scraped"].value_counts()

last_scraped
2026-09-18    12943
2026-09-17     2291
2026-09-19        2
Name: count, dtype: int64

▶️ Print out the DataFrame's summary.


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15236 entries, 0 to 15235
Data columns (total 90 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            15236 non-null  int64  
 1   listing_url                                   15236 non-null  str    
 2   scrape_id                                     15236 non-null  int64  
 3   last_scraped                                  15236 non-null  str    
 4   source                                        3694 non-null   str    
 5   name                                          15236 non-null  str    
 6   description                                   15192 non-null  str    
 7   neighborhood_overview                         0 non-null      float64
 8   picture_url                                   15236 non-null  str    
 9   host_id                                       15236 non-null  int64  
 1

▶️ Although `info()` provides some information about the number of non-missing values, you can explicitly count missing values in each column using the following code:


In [6]:
df.isna().sum()
# Equivalent to:
df.isnull().sum()

id                                                  0
listing_url                                         0
scrape_id                                           0
last_scraped                                        0
source                                          11542
                                                ...  
calculated_host_listings_count                      0
calculated_host_listings_count_entire_homes         0
calculated_host_listings_count_private_rooms        0
calculated_host_listings_count_shared_rooms         0
reviews_per_month                                3301
Length: 90, dtype: int64

To only filter columns that have missing values, you can use:


In [7]:
df.isna().sum()[df.isna().sum() > 0]
# Equivalent to:
df.isnull().sum()[df.isnull().sum() > 0]

source                         11542
description                       44
neighborhood_overview          15236
host_since                     15236
hosts_time_as_user_years           4
hosts_time_as_user_months          4
hosts_time_as_host_years           4
hosts_time_as_host_months          4
host_location                   3188
host_about                      4804
host_response_time             15236
host_response_rate             15236
host_acceptance_rate           15236
host_thumbnail_url             15236
host_neighbourhood             15236
host_total_listings_count      15236
host_verifications             15236
neighbourhood                  15236
bathrooms                      15236
bathrooms_text                     9
bedrooms                           2
beds                             414
price                           2626
price_quote_checkin_date        1820
price_quote_checkout_date       1820
price_quote_total_price         2792
price_quote_price_per_night     2792
p

### Select and rename columns

We keep the columns we need for a typical price analysis. Note that we select `bathrooms_text` rather than `bathrooms`: in this snapshot, the numeric `bathrooms` column is empty for every listing. Inside Airbnb's data dictionary explains that the bathrooms field on Airbnb's website changed from a number to a text description, and that the numeric column is used for older scrapes.

In [8]:
df_c = df[
    [
        "name",
        "neighbourhood_cleansed",
        "room_type",
        "bedrooms",
        "bathrooms_text",
        "accommodates",
        "minimum_nights",
        "price",
        "availability_365",
        "number_of_reviews",
        "review_scores_rating",
        "latitude",
        "longitude",
        "host_is_superhost",
    ]
].copy()

df_c = df_c.rename(
    columns={
        "neighbourhood_cleansed": "neighbourhood",
        "review_scores_rating": "review_score",
        "host_is_superhost": "is_superhost",
    },
)

df_c.head(2)

,name,neighbourhood,room_type,bedrooms,bathrooms_text,accommodates,minimum_nights,price,availability_365,number_of_reviews,review_score,latitude,longitude,is_superhost
0,Huge flat for 8 people close to Sagrada Familia,la Sagrada Família,Entire home/apt,3.0,2 baths,8,1.0,$274.00,162,60,4.42,41.40556,2.17262,f
1,CCIB Forum • Spacious 3BR • Large Balcony,el Besòs i el Maresme,Entire home/apt,3.0,2 baths,5,3.0,$407.67,128,102,4.84,41.41243,2.21975,t


### Parse the number of bathrooms from text

Check that the numeric `bathrooms` column really is empty, and look at the most common text values.

In [9]:
print("Share of listings with no numeric bathrooms value:", df["bathrooms"].isna().mean())

df_c["bathrooms_text"].value_counts().head(10)

Share of listings with no numeric bathrooms value: 1.0


bathrooms_text
1 bath              6193
2 baths             3227
1 shared bath       1554
1 private bath      1396
1.5 baths            538
2 shared baths       464
0 baths              375
3 baths              297
0 shared baths       213
1.5 shared baths     210
Name: count, dtype: int64

Most values start with a number (`"1 bath"`, `"1.5 shared baths"`). A few say `"Half-bath"`, `"Shared half-bath"`, or `"Private half-bath"`, which we count as 0.5. The regular expression `(\d+(?:\.\d+)?)` extracts the first number, including decimals. Listings with no text keep a missing value.

In [10]:
bath_text = df_c["bathrooms_text"].str.lower()

df_c["bathrooms"] = np.where(
    bath_text.str.contains("half-bath", na=False),
    0.5,
    bath_text.str.extract(r"(\d+(?:\.\d+)?)", expand=False).astype(float),
)

# Check the mapping from text to number, then drop the text column.
display(df_c[["bathrooms_text", "bathrooms"]].value_counts(dropna=False).head(12))
df_c = df_c.drop(columns="bathrooms_text")

bathrooms_text    bathrooms
1 bath            1.0          6193
2 baths           2.0          3227
1 shared bath     1.0          1554
1 private bath    1.0          1396
1.5 baths         1.5           538
2 shared baths    2.0           464
0 baths           0.0           375
3 baths           3.0           297
0 shared baths    0.0           213
1.5 shared baths  1.5           210
4 baths           4.0           152
2.5 baths         2.5           131
Name: count, dtype: int64

:::{tip} Extension

The text also says whether a bathroom is shared or private. If that matters for your analysis, you could create a `shared_bathroom` flag with `bath_text.str.contains("shared")` before dropping the text column.

:::

### Parse price as `float`s

Inside Airbnb's data dictionary describes `price` as the daily price in local currency (euros, for Barcelona) and notes that the `$` sign is a technical artifact of the export.

In [11]:
df_c["price"] = (
    df_c["price"].str.replace("$", "").str.replace(",", "").astype(np.float64)
)

### Convert `is_superhost` to 0s and 1s

`host_is_superhost` is stored as `"t"` (true) or `"f"` (false). A tempting shortcut is `np.where(df_c["is_superhost"] == "t", 1, 0)`, but it turns every **missing** value into 0, which silently claims that the host is not a superhost. Instead, we map `"t"` to 1 and `"f"` to 0, so missing values stay missing. The nullable integer type `"Int64"` can hold whole numbers and missing values in the same column. (An earlier snapshot of this data had a few hundred missing superhost values.)

In [12]:
df_c["is_superhost"] = df_c["is_superhost"].map({"t": 1, "f": 0}).astype("Int64")

df_c["is_superhost"].value_counts(dropna=False)

is_superhost
0    11385
1     3851
Name: count, dtype: Int64

### Handle missing values

Count the missing values in the selected columns.

In [13]:
df_c.isna().sum()

name                    0
neighbourhood           0
room_type               0
bedrooms                2
accommodates            0
minimum_nights       2045
price                2626
availability_365        0
number_of_reviews       0
review_score         3301
latitude                0
longitude               0
is_superhost            0
bathrooms               9
dtype: int64

Before deciding what to do, look at _why_ values are missing. The cell below checks two patterns: whether listings without a price are open for booking, and whether a missing `review_score` simply means the listing has no reviews.

In [14]:
no_price = df_c["price"].isna()
no_reviews = df_c["number_of_reviews"] == 0

print(f"Listings with no price: {no_price.sum():,}")
print(
    "  share with no open dates in the next 365 days: "
    f"{(df_c.loc[no_price, 'availability_365'] == 0).mean():.0%}"
)
print(f"Listings with no reviews: {no_reviews.sum():,}")
print(
    "  share with a missing review_score: "
    f"{df_c.loc[no_reviews, 'review_score'].isna().mean():.0%}"
)
print(
    "Listings with at least one review but no review_score: "
    f"{(df_c['review_score'].isna() & ~no_reviews).sum()}"
)

Listings with no price: 2,626
  share with no open dates in the next 365 days: 62%
Listings with no reviews: 3,301
  share with a missing review_score: 100%
Listings with at least one review but no review_score: 0


Each column with missing values needs its own decision:

- **`price`** is missing for about 17% of listings, and most of those have no open dates in the next year. (Inside Airbnb's data dictionary notes that a date can be unavailable because a guest booked it or the host blocked it.) Price is the main variable for most analyses of this data, and filling it in would invent the very thing we want to study. We **drop** these rows.
- **`bedrooms`** and **`bathrooms`** are missing for only a handful of listings. We **drop** these rows too.
- **`review_score`** is missing for exactly the listings that have no reviews yet. Dropping them would remove every new or never-booked listing (more than a fifth of the data) and tilt the sample toward established listings. Filling in the average score would pretend that unreviewed listings are average. We **keep** these rows and leave `review_score` missing; `number_of_reviews == 0` identifies them.
- **`minimum_nights`** is missing for about 2,000 listings. The column summary above shows that the same number of listings is missing every minimum and maximum nights column, so the booking rules were not captured for them in this scrape. It is not essential for a price analysis, so we **keep** these rows and leave the value missing. We also store it as a nullable integer (`"Int64"`).

:::{important} Drop, fill, or keep missing?

There is no single right answer. Drop rows when the value is essential and cannot be recovered. Fill in a value only when you have a defensible rule (for example, a known default). Keep the value missing when the absence itself carries information, as with listings that have no reviews. Whatever you choose, write it down, and check how many rows each choice removes.

:::

In [15]:
df_c["bathrooms"].unique()

array([ 2. ,  3. ,  1. ,  3.5,  1.5,  2.5,  4. ,  4.5,  0. ,  6. ,  nan,
        5. ,  5.5,  0.5,  7. ,  8. , 10. ,  9. , 12. , 13. , 14. , 16. ,
       11. ])

In [16]:
df_c["bedrooms"].unique()

array([ 3.,  4.,  1.,  2.,  0.,  7.,  5.,  6.,  8.,  9., 12., nan, 10.,
       16., 11., 20., 24., 15., 14., 18., 26., 29.])

In [17]:
n_before = len(df_c)

df_c = df_c.dropna(subset=["price", "bedrooms", "bathrooms"])
df_c["minimum_nights"] = df_c["minimum_nights"].astype("Int64")

print(f"Kept {len(df_c):,} of {n_before:,} listings ({len(df_c) / n_before:.0%}).")

Kept 12,601 of 15,236 listings (83%).


In [18]:
df_c["room_type"].value_counts()

room_type
Entire home/apt    9385
Private room       3039
Shared room         110
Hotel room           67
Name: count, dtype: int64

### Validate the cleaned data

Cleaning is not finished until you check the result. The checks below state rules that every row should meet. Each prints `True` if the rule holds for all rows. Missing values are allowed where we decided to keep them.

In [19]:
checks = {
    "price is positive": (df_c["price"] > 0).all(),
    "no missing price, bedrooms, or bathrooms": df_c[["price", "bedrooms", "bathrooms"]]
    .notna()
    .all()
    .all(),
    "review_score is between 0 and 5": df_c["review_score"].dropna().between(0, 5).all(),
    "minimum_nights is at most 365": (df_c["minimum_nights"].dropna() <= 365).all(),
    "coordinates are in the Barcelona area": (
        df_c["latitude"].between(41.3, 41.5).all()
        and df_c["longitude"].between(2.0, 2.3).all()
    ),
}

for rule, passed in checks.items():
    print(f"{passed!s:5}  {rule}")

True   price is positive
True   no missing price, bedrooms, or bathrooms
True   review_score is between 0 and 5
True   minimum_nights is at most 365
True   coordinates are in the Barcelona area


Rules catch impossible values. Plausible but extreme values need a look as well. `describe()` shows the range of each numeric column, and sorting by price shows the most expensive listings.

In [20]:
df_c[["price", "minimum_nights", "accommodates", "bedrooms", "bathrooms"]].describe()

,price,minimum_nights,accommodates,bedrooms,bathrooms
count,12601.000000,10725.0,12601.000000,12601.000000,12601.000000
mean,253.442013,13.644569,3.880644,2.006984,1.450083
std,386.443737,16.165803,2.579044,1.555823,1.006313
min,12.810000,1.0,1.000000,0.000000,0.000000
25%,93.440000,1.0,2.000000,1.000000,1.000000
50%,190.670000,2.0,4.000000,2.000000,1.000000
75%,284.000000,31.0,5.000000,3.000000,2.000000
max,11245.030000,250.0,16.000000,29.000000,16.000000


In [21]:
df_c.sort_values("price", ascending=False)[
    ["name", "room_type", "accommodates", "bedrooms", "price", "number_of_reviews"]
].head(8)

,name,room_type,accommodates,bedrooms,price,number_of_reviews
7780,Amazing loft-style apartment in Las Ramblas,Entire home/apt,2,1.0,11245.03,0
14397,Comfy 18BR GroupFlat for 36ppl - Sagrada Familia,Entire home/apt,16,18.0,7679.00,0
11790,"web Acomodis, Apartamento familiar",Entire home/apt,16,12.0,7293.00,0
11821,Group Flat 14-Bedroom for 27ppl | Passeig Gràcia,Entire home/apt,16,14.0,6304.00,0
10979,Elegant 2-bedr Apartment with 1-bathr 4 people,Entire home/apt,16,2.0,6129.00,0
9863,Just 10 minutes from Casa Batlló Gaudi - 22 pe...,Entire home/apt,16,10.0,5981.00,0
11959,Ideal Spot: Great 29-Bedrooms Group Flat-54 pe...,Entire home/apt,16,29.0,5930.00,0
14396,Great 15BR GroupFlat for 30ppl - Sagrada Familia,Entire home/apt,16,15.0,5792.00,0


All rules pass, but the extremes deserve a second look. The eight most expensive listings, priced from about €5,800 to €11,245 a night, have no reviews at all. Some are large group flats where a high price is plausible. Others look like placeholder prices, for example a one-bedroom loft for two people listed at €11,245 a night. Note also that `accommodates` stops at 16, so it cannot tell a 16-person flat from a 36-person one.

We keep these listings in the saved file because we cannot prove they are wrong. An analysis of typical prices should use the median (about €190 a night here) rather than the mean (about €253), or state a rule for excluding such listings.

### Check the cleaned DataFrame


In [22]:
df_c.info()

<class 'pandas.DataFrame'>
Index: 12601 entries, 0 to 15235
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   name               12601 non-null  str    
 1   neighbourhood      12601 non-null  str    
 2   room_type          12601 non-null  str    
 3   bedrooms           12601 non-null  float64
 4   accommodates       12601 non-null  int64  
 5   minimum_nights     10725 non-null  Int64  
 6   price              12601 non-null  float64
 7   availability_365   12601 non-null  int64  
 8   number_of_reviews  12601 non-null  int64  
 9   review_score       10120 non-null  float64
 10  latitude           12601 non-null  float64
 11  longitude          12601 non-null  float64
 12  is_superhost       12601 non-null  Int64  
 13  bathrooms          12601 non-null  float64
dtypes: Int64(2), float64(6), int64(3), str(3)
memory usage: 2.3 MB


In [23]:
df_c.head()

,name,neighbourhood,room_type,bedrooms,accommodates,minimum_nights,price,availability_365,number_of_reviews,review_score,latitude,longitude,is_superhost,bathrooms
0,Huge flat for 8 people close to Sagrada Familia,la Sagrada Família,Entire home/apt,3.0,8,1,274.00,162,60,4.42,41.40556,2.17262,0,2.0
1,CCIB Forum • Spacious 3BR • Large Balcony,el Besòs i el Maresme,Entire home/apt,3.0,5,3,407.67,128,102,4.84,41.41243,2.21975,1,2.0
2,VIDRE HOME PLAZA REAL on LAS RAMBLAS,el Barri Gòtic,Entire home/apt,4.0,9,1,545.60,316,314,4.60,41.37978,2.17623,1,3.0
3,In front of the beach,la Barceloneta,Private room,1.0,1,31,44.47,77,108,4.79,41.38043,2.19094,0,1.0
4,Architect's Loft near Gaudí & Passeig de Gràcia,la Dreta de l'Eixample,Entire home/apt,1.0,4,1,187.46,283,464,4.56,41.39631,2.16832,1,1.0


### Save the cleaned data

In [24]:
df_c.to_csv("cleaned-output.csv", index=None)

## Summary

In this notebook you:

- Loaded 15,236 Barcelona listings with 90 columns from the Inside Airbnb snapshot of 17 September 2026.
- Selected 14 columns, parsed the number of bathrooms from `bathrooms_text` (the numeric column is empty in this snapshot), converted `price` to numbers, and converted the superhost flag to 1/0 without turning missing values into 0.
- Dropped the listings without a price, bedrooms, or bathrooms, keeping 12,601 listings (83%). Kept never-reviewed listings with a missing `review_score`, and kept missing `minimum_nights` values.
- Validated the result with explicit rules, and found extreme prices among never-reviewed listings, which we kept and documented.
- Saved the cleaned data to `cleaned-output.csv`.